# Rule-based parsing: regex to templates

**Module 2 -- Lesson 6**

In the previous notebook, reconstructing RFC structure from PDF-extracted text recovered ~82% field coverage using `email.parser`. That leaves ~18% of files where the reconstruction assumptions don't hold -- unusual boilerplate, OCR damage, missing headers.

This notebook takes a different approach: instead of restructuring text to fit a parser, use regex patterns to match the text as it is.

1. **Regex primer** -- the five patterns used throughout
2. **Load corpus + strip boilerplate** -- preparing the text
3. **Header structure** -- alternating labels and values, with field variations
4. **Greedy extraction** -- walk the lines, grab what you recognize
5. **Where it produces wrong data** -- OCR-corrupted labels and silent failures
6. **So what can we rely on?** -- emails of the same type share an exact role sequence
7. **Templates** -- encode exact role sequences as data
8. **Templates as rejection tools** -- match exactly or fail cleanly
9. **How the 3 starter templates were built**
10. **Running on the full corpus** -- 3 templates, ~98.6% coverage
11. **Build your own template** -- exercise

## 1. Regex Primer

A regular expression (regex) is a small pattern language for finding text. You write a pattern — like `^From:\s+(.+)` — and the regex engine scans your text looking for anything that matches it. Patterns are built from literal characters that match themselves and special symbols that match *classes* of characters (any digit, any whitespace) or control *where* and *how many times* something can appear.

If you've never used regex before, don't worry — the five examples below cover everything you'll need for this notebook. Run the cell and read the output.

In [1]:
import re

# --- Pattern 1: ^ anchors to the start of a line ---
print("Pattern: ^From")
print(re.match(r'^From', 'From: Matthew Lenhart'))   # match
print(re.match(r'^From', '  From: Matthew Lenhart')) # None — leading space
print()

# --- Pattern 2: $ anchors to the end (empty label line) ---
print("Pattern: ^From:\\s*$")
print(bool(re.match(r'^From:\s*$', 'From:')))         # True — label-only
print(bool(re.match(r'^From:\s*$', 'From: Matthew'))) # False — has value
print()

# --- Pattern 3: \s* matches zero or more whitespace ---
print("Pattern: ^From:\\s+(.+)")
m = re.match(r'^From:\s+(.+)', 'From:   Matthew Lenhart')
print(m.group(1) if m else None)  # 'Matthew Lenhart'
print()

# --- Pattern 4: \S+ matches one or more non-whitespace ---
print("Pattern: \\S+ to find email addresses")
m = re.search(r'\S+@\S+', 'To: vapte@yahoo.com @ ENRON')
print(m.group(0) if m else None)  # 'vapte@yahoo.com'
print()

# --- Pattern 5: re.IGNORECASE ---
print("Pattern: re.IGNORECASE")
print(bool(re.match(r'^FROM:', 'From:', re.IGNORECASE)))  # True
print(bool(re.match(r'^from:', 'FROM:', re.IGNORECASE)))  # True

Pattern: ^From
<re.Match object; span=(0, 4), match='From'>
None

Pattern: ^From:\s*$
True
False

Pattern: ^From:\s+(.+)
Matthew Lenhart

Pattern: \S+ to find email addresses
vapte@yahoo.com

Pattern: re.IGNORECASE
True
True


## 2. Load Corpus + Strip Boilerplate

Each PDF-extracted text file starts with government processing stamps — `CONFIDENTIAL`, `Enron Corp.`, `Case No.`, `Doc No.`, `Date:`, `ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.`, `RELEASE IN FULL` or `RELEASE IN PART` — that must be removed before we can see the email header. Run the cell and compare before and after.

In [2]:
import json
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm

TXT_DIR = Path("data/extracted_text")
txt_files = sorted(TXT_DIR.glob("*.txt"))
print(f"Corpus size: {len(txt_files):,} files")

BOILERPLATE = [
    r"^'?CONFIDENTIAL.*$",
    r"^'?UNCLASSIFIED.*$",
    r"^Enron\s*Corp.*$",
    r"^Case\s*No[,\.].*$",
    r"^Doc\s*No[,\.].*$",
    r"^Doe\s*No.*$",
    r"^Dot\s*No.*$",
    r"^Pate:.*$",
    r"^Date:?\s*\d",
    r"^ENRON\s*CORP.*$",
    r"^SUBJECT TO PROTECTIVE.*$",
    r"^RELEASE IN.*$",
    r"^ENRON[-=]\d+.*$",
    r"^ENRON-.*$",
    r"^EC-2002.*$",
]


def strip_boilerplate(text):
    """Remove government processing stamps from extracted text."""
    lines = text.split("\n")
    cleaned = [
        line for line in lines
        if not any(re.match(p, line.strip(), re.IGNORECASE) for p in BOILERPLATE)
    ]
    return "\n".join(cleaned).strip()


# Demo: before and after on a real file (E0048ADF3 — alternating layout)
sample_raw = txt_files[1].read_text(encoding="utf-8")  # E0048ADF3.txt
sample_clean = strip_boilerplate(sample_raw)

print("\n=== BEFORE (first 400 chars) ===")
print(sample_raw[:400])
print("\n=== AFTER ===")
print(sample_clean[:400])

Corpus size: 4,911 files

=== BEFORE (first 400 chars) ===
CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0000D1D0
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
CONFIDENTIAL - SUBJECT TO PROTECTIVE ORDER
From:
EDIS Email Service <edismail@incident.com>
Sent:
Fri, 5 Apr 2002 00:07:00 -0800 (PST)
To:
Motley, Matt <matt.motley@enron.com>
Subject:
[EDIS]  EQ 4 2 SAN 

=== AFTER ===
From:
EDIS Email Service <edismail@incident.com>
Sent:
Fri, 5 Apr 2002 00:07:00 -0800 (PST)
To:
Motley, Matt <matt.motley@enron.com>
Subject:
[EDIS]  EQ 4 2 SAN BERNARDINO COUNTY [News: Statewide]
PRELIMINARY EVENT REPORT
Southern California Seismic Network (TriNet) operated by Caltech and USGS
Version 1: This report supersedes any earlier reports about this event.
This is a computer generated sol


/Users/henryadamcollie/Documents/GitHub/1_pdf_to_kg_notebooks/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Header Structure

Now that boilerplate is gone, we can look at what the email headers actually look like. Run the next cell to print the first 10 lines of a stripped file.

In [3]:
# Show the header structure of a sample file
sample = strip_boilerplate(txt_files[1].read_text(encoding="utf-8"))

print("Header structure (first file after stripping):")
for i, line in enumerate(sample.split("\n")[:10]):
    print(f"  [{i}] {line.rstrip()!r}")

Header structure (first file after stripping):
  [0] 'From:'
  [1] 'EDIS Email Service <edismail@incident.com>'
  [2] 'Sent:'
  [3] 'Fri, 5 Apr 2002 00:07:00 -0800 (PST)'
  [4] 'To:'
  [5] 'Motley, Matt <matt.motley@enron.com>'
  [6] 'Subject:'
  [7] '[EDIS]  EQ 4 2 SAN BERNARDINO COUNTY [News: Statewide]'
  [8] 'PRELIMINARY EVENT REPORT'
  [9] 'Southern California Seismic Network (TriNet) operated by Caltech and USGS'


Notice the alternating pattern: `From:` on its own line, then the value on the next line, then `Sent:` on its own line, and so on.

Not every email has the same fields — some include `Cc:`, others skip `Subject:`. Run the next cell to count how many emails have each field combination.

In [4]:
# For each file, find which header labels appear before the body starts
LABEL_RE = re.compile(r'^(From|Sent|To|Cc|Subject):\s*$', re.IGNORECASE)

field_combos = Counter()
no_labels = 0

for f in txt_files:
    cleaned = strip_boilerplate(f.read_text(encoding="utf-8"))
    labels = [LABEL_RE.match(line.strip()).group(1).capitalize()
              for line in cleaned.split("\n")[:20]
              if LABEL_RE.match(line.strip())]

    if labels:
        field_combos["/".join(labels)] += 1
    else:
        no_labels += 1

total = len(txt_files)
print(f"Files with recognisable labels: {total - no_labels:,} / {total:,}")
print(f"Files without (OCR-garbled):    {no_labels:,} / {total:,}")
print()
for combo, count in field_combos.most_common(10):
    print(f"  {combo:35s}: {count:,}  ({count/total*100:.1f}%)")

Files with recognisable labels: 4,116 / 4,911
Files without (OCR-garbled):    795 / 4,911

  From/Sent/To/Subject               : 2,926  (59.6%)
  From/Sent/To/Cc/Subject            : 474  (9.7%)
  From/Sent/To/Subject/Cc            : 290  (5.9%)
  From/Sent/To                       : 77  (1.6%)
  From/Sent/To/Subject/From          : 64  (1.3%)
  From/Sent/To/Subject/From/Sent/To/Subject: 41  (0.8%)
  From/Sent/To/Cc/Subject/Cc         : 36  (0.7%)
  From/Sent/To/Subject/From/Sent/To  : 33  (0.7%)
  From/Sent/To/Subject/From/Sent     : 23  (0.5%)
  From/Sent/To/Cc                    : 23  (0.5%)


## 4. Greedy extraction

The alternating pattern is predictable enough to write a simple extractor: walk the lines top to bottom, and whenever you see a label you recognize (`From:`, `Sent:`, etc.), record the next non-empty line as that field's value. Everything between one label and the next gets absorbed into the current field.

This approach is **greedy** — it always produces a result. It never says "I don't know" or "this doesn't look right." It just keeps absorbing lines until the next recognized label appears.

Run the first cell to define the function, and the second to test it on three files.

In [5]:
def parse_alternating(text):
    """Extract fields from alternating layout (label line, value next line)."""
    fields = {}
    current_label = None
    in_body = False

    for line in text.split("\n"):
        stripped = line.strip()
        if in_body:
            break

        m = re.match(r'^(From|Sent|To|Cc|Subject):\s*$', stripped, re.IGNORECASE)
        if m:
            current_label = m.group(1).lower()
            fields[current_label] = ""
        elif current_label and stripped:
            fields[current_label] += (" " if fields[current_label] else "") + stripped
            if current_label == "subject":
                in_body = True

    return fields

In [6]:
# Test on three files
for f in [txt_files[1], txt_files[7], txt_files[12]]:
    text = f.read_text(encoding="utf-8")
    fields = parse_alternating(strip_boilerplate(text))
    print(f"=== {f.name} ===")
    for k in ("from", "sent", "to", "cc", "subject"):
        val = fields.get(k, "")
        print(f"  {k:8s}: {val[:80]!r}")
    print()

=== E0000D1D0.txt ===
  from    : 'EDIS Email Service <edismail@incident.com>'
  sent    : 'Fri, 5 Apr 2002 00:07:00 -0800 (PST)'
  to      : 'Motley, Matt <matt.motley@enron.com>'
  cc      : ''
  subject : '[EDIS]  EQ 4 2 SAN BERNARDINO COUNTY [News: Statewide]'

=== E004FC63F.txt ===
  from    : 'jdasovic@enron.com <jdasovic@enron.com>'
  sent    : 'Fri, 1 Sep 2000 03:30:00 -0700 (PDT)'
  to      : 'Steven.J.Kean@enron.com <steven.j.kean@enron.com>'
  cc      : ''
  subject : 'Re: California Lawmakers Vote to Limit Power Costs - WSJ'

=== E00825F58.txt ===
  from    : 'Grigsby, Mike <mike.grigsby@enron.com>'
  sent    : 'Mon, 1 Oct 2001 21:07:05 -0700 (PDT)'
  to      : 'Holst, Keith <keith.holst@enron.com>, Reitmeyer, Jay <jay.reitmeyer@enron.com>'
  cc      : ''
  subject : 'FW: New Gen Report'



In [7]:
# Field coverage on the first 50 digital files (clean boilerplate)
digital_files = [
    f for f in txt_files
    if f.read_text(encoding="utf-8").startswith("CONFIDENTIAL")
][:50]

counts = {k: 0 for k in ("from", "sent", "to", "cc", "subject")}
for f in digital_files:
    text = f.read_text(encoding="utf-8")
    fields = parse_alternating(strip_boilerplate(text))
    for k in counts:
        if fields.get(k):
            counts[k] += 1

print(f"Field coverage on first 50 digital emails:")
for k, n in counts.items():
    print(f"  {k:8s}: {n}/50")

Field coverage on first 50 digital emails:
  from    : 50/50
  sent    : 50/50
  to      : 45/50
  cc      : 6/50
  subject : 50/50


## 5. Where it produces incorrect data

The simple extractor works well on clean digital files — and at first glance it appears to work on OCR files too. That's the problem.

OCR can corrupt labels: `Ce:` instead of `Cc:`, `Toa:` instead of `To:`. The extractor doesn't recognize these as labels, so it treats them as value lines and silently appends them to the previous field. The result looks plausible — `From`, `Sent`, and `Subject` are all correct — but the `To` field has quietly absorbed `Ce:` and the Cc recipients, and the `Cc` field is missing entirely.

At scale, across thousands of files, you have no way to tell which records are clean and which are subtly wrong.

Run the cell to see this in action.

In [8]:
# E25B1E9BF has "Ce:" instead of "Cc:" — a real OCR corruption
ocr_file = TXT_DIR / "E25B1E9BF.txt"
ocr_cleaned = strip_boilerplate(ocr_file.read_text(encoding="utf-8"))

print("Header lines (E25B1E9BF):")
for i, line in enumerate(ocr_cleaned.split("\n")[:12]):
    print(f"  [{i:2d}] {line.rstrip()!r}")

print()
result = parse_alternating(ocr_cleaned)
print("parse_alternating result:")
for k in ("from", "sent", "to", "cc", "subject"):
    v = result.get(k, "(missing)")
    if len(str(v)) > 90:
        v = str(v)[:90] + "..."
    print(f"  {k:8s}: {v!r}")

Header lines (E25B1E9BF):
  [ 0] 'From:'
  [ 1] 'Hitchcock, Dorie <dorie.hitchcock@enron.com>'
  [ 2] 'Sent:'
  [ 3] 'Fri, 20 Jul 2001 11:13:22 -0700 (PDT)'
  [ 4] 'To:'
  [ 5] 'Kitchen, Louise <louise.kitchen@enron.com>, Lavorato, John'
  [ 6] '<john.lavorato@enron.com>'
  [ 7] 'Ce:'
  [ 8] 'Schoppe, Tammie <tammie.schoppe@enron.com>, Hillis, Kimberly'
  [ 9] '<kimberly.hillis@enron.com>'
  [10] 'Subject:'
  [11] 'EA EVENT MASTER SCHEDULE'

parse_alternating result:
  from    : 'Hitchcock, Dorie <dorie.hitchcock@enron.com>'
  sent    : 'Fri, 20 Jul 2001 11:13:22 -0700 (PDT)'
  to      : 'Kitchen, Louise <louise.kitchen@enron.com>, Lavorato, John <john.lavorato@enron.com> Ce: S...'
  cc      : '(missing)'
  subject : 'EA EVENT MASTER SCHEDULE'


Look at line 7: `Ce:` is OCR's misreading of `Cc:`. The extractor doesn't recognize it as a label, so it treats it as a continuation of the `To` value — silently appending `Ce:` and the Cc recipients into the wrong field.

From, Sent, and Subject are all correct. At a glance the output looks fine. But the `To` field is polluted and `Cc` is missing entirely. Across thousands of files, there is no way to tell which records have this problem without inspecting each one.

## 6. So what can we rely on?

The greedy extractor fails because it's permissive — it always produces a result, even when that result is wrong. To build something stricter, we first need to understand what's actually consistent in the data.

Each line in an email header does one of three jobs. It might be a label on its own (`From:`), a value on its own (`Matthew Lenhart`), or a label and value together on the same line (`From: Matthew Lenhart`). Let's call this the line's **role**.

Run the cell below. It walks through several emails and prints the role of each header line. Look at the output and compare emails that have the same fields — you'll see they produce the exact same sequence of roles, every time.

In [9]:
HEADER_LABELS = {"from", "sent", "to", "cc", "subject", "attachments"}


def classify_line(line):
    """Assign a role to a single (non-empty) line."""
    stripped = line.strip()

    # Label-only line
    if re.match(
        r'^(From|Sent|To|Cc|Subject|Attachments?):\s*$', stripped, re.IGNORECASE
    ):
        label = re.match(r'^(\w+):', stripped).group(1).lower()
        return ("LABEL", label)

    # Same-line (label + value)
    m = re.match(
        r'^(From|Sent|To|Cc|Subject|Attachments?):\s+(\S.+)', stripped, re.IGNORECASE
    )
    if m:
        label = m.group(1).lower()
        return ("LABEL_VALUE", label)

    return ("VALUE", None)


def role_sequence(text, max_lines=12):
    """Return the sequence of roles for the first few non-empty header lines."""
    seq = []
    for line in text.split("\n"):
        if not line.strip():
            continue
        role, label = classify_line(line)
        tag = f"{role}({label})" if label else role
        seq.append(tag)
        if len(seq) >= max_lines:
            break
        if label == "subject" and role in ("LABEL_VALUE",):
            break
    return seq


The two functions above are the building blocks. `classify_line` assigns a role to any single line, and `role_sequence` walks a stripped email and returns the full sequence.

Run the next cell to see these in action across the corpus. Compare the sequences — emails with the same fields produce the same sequence every time.

In [10]:
# Show role sequences for 5 From/Sent/To/Subject files
print("=== From/Sent/To/Subject emails ===")
fstx_count = 0
for f in txt_files:
    text = f.read_text(encoding="utf-8")
    cleaned = strip_boilerplate(text)
    seq = role_sequence(cleaned)
    seq_str = " -> ".join(seq[:10])
    if "LABEL(from)" in seq_str and "LABEL(subject)" in seq_str and "LABEL(cc)" not in seq_str:
        print(f"  {f.stem}: {seq_str}")
        fstx_count += 1
        if fstx_count >= 5:
            break

# Show role sequences for 3 From/Sent/To/Cc/Subject files
print()
print("=== From/Sent/To/Cc/Subject emails ===")
fstcx_count = 0
for f in txt_files:
    text = f.read_text(encoding="utf-8")
    cleaned = strip_boilerplate(text)
    seq = role_sequence(cleaned)
    seq_str = " -> ".join(seq[:12])
    if "LABEL(cc)" in seq_str:
        print(f"  {f.stem}: {seq_str}")
        fstcx_count += 1
        if fstcx_count >= 3:
            break

print()
print("Every email with the same fields produces the identical role sequence.")

=== From/Sent/To/Subject emails ===
  E0000D1D0: LABEL(from) -> VALUE -> LABEL(sent) -> VALUE -> LABEL(to) -> VALUE -> LABEL(subject) -> VALUE -> VALUE -> VALUE
  E004FC63F: LABEL(from) -> VALUE -> LABEL(sent) -> VALUE -> LABEL(to) -> VALUE -> LABEL(subject) -> VALUE -> VALUE -> VALUE
  E0055EC9B: LABEL(from) -> VALUE -> LABEL(sent) -> VALUE -> LABEL(to) -> LABEL(subject) -> VALUE -> VALUE -> VALUE -> VALUE
  E0058489E: LABEL(from) -> VALUE -> LABEL(sent) -> VALUE -> LABEL(to) -> VALUE -> LABEL(subject) -> VALUE -> VALUE -> LABEL_VALUE(from)
  E00825F58: LABEL(from) -> VALUE -> LABEL(sent) -> VALUE -> LABEL(to) -> VALUE -> VALUE -> LABEL(subject) -> VALUE -> VALUE

=== From/Sent/To/Cc/Subject emails ===
  E0000813E: LABEL(from) -> VALUE -> LABEL(sent) -> VALUE -> LABEL(to) -> VALUE -> LABEL(cc) -> VALUE -> LABEL(subject) -> VALUE -> VALUE -> VALUE
  E000B745B: LABEL(from) -> VALUE -> LABEL(sent) -> VALUE -> LABEL(to) -> VALUE -> LABEL(cc) -> VALUE -> LABEL(subject) -> VALUE -> VALUE ->

## 7. Templates

Since every email with the same fields produces the same role sequence, we can write that sequence down and use it as a **template**. 

A template is a list of expected steps — "first I expect a bare `From:` label, then a value line, then a bare `Sent:` label, then a value line..." and so on. 

A matching function walks the template against the actual text, step by step. If every step matches, it extracts the fields. If any step fails, it returns `None` — no partial results, no guessing.

Run the next two cells to define three templates and the generic matcher that walks them.

In [11]:
from enum import Enum, auto


class Role(Enum):
    LABEL = auto()        # bare label line: 'From:'
    VALUE = auto()        # data line (non-label)
    LABEL_VALUE = auto()  # label + value on same line: 'From: Matthew'
    BODY = auto()         # first body line — stop here


L, V, LV, B = Role.LABEL, Role.VALUE, Role.LABEL_VALUE, Role.BODY


# Template for alternating From/Sent/To/Subject (no Cc) — most common
TEMPLATE_ALT_FSTX = {
    "name": "alt_FStX",
    "structure": [
        (L, "from"), (V, "from"),
        (L, "sent"), (V, "sent"),
        (L, "to"),   (V, "to"),
        (L, "subject"), (V, "subject"),
        (B, None),
    ],
}

# Template for alternating From/Sent/To/Cc/Subject — second most common
TEMPLATE_ALT_FSTCX = {
    "name": "alt_FStCX",
    "structure": [
        (L, "from"), (V, "from"),
        (L, "sent"), (V, "sent"),
        (L, "to"),   (V, "to"),
        (L, "cc"),   (V, "cc"),
        (L, "subject"), (V, "subject"),
        (B, None),
    ],
}

# Template for alternating From/Sent/To(empty)/Subject — To label but no value
TEMPLATE_ALT_FS_X = {
    "name": "alt_Fs_X",
    "structure": [
        (L, "from"), (V, "from"),
        (L, "sent"), (V, "sent"),
        (L, "to"),               # To label with no value
        (L, "subject"), (V, "subject"),
        (B, None),
    ],
}

print("Three templates defined:")
for t in [TEMPLATE_ALT_FSTX, TEMPLATE_ALT_FSTCX, TEMPLATE_ALT_FS_X]:
    steps = [f"{r.name}({f})" for r, f in t["structure"]]
    print(f"  {t['name']:12s}: {' -> '.join(steps)}")

Three templates defined:
  alt_FStX    : LABEL(from) -> VALUE(from) -> LABEL(sent) -> VALUE(sent) -> LABEL(to) -> VALUE(to) -> LABEL(subject) -> VALUE(subject) -> BODY(None)
  alt_FStCX   : LABEL(from) -> VALUE(from) -> LABEL(sent) -> VALUE(sent) -> LABEL(to) -> VALUE(to) -> LABEL(cc) -> VALUE(cc) -> LABEL(subject) -> VALUE(subject) -> BODY(None)
  alt_Fs_X    : LABEL(from) -> VALUE(from) -> LABEL(sent) -> VALUE(sent) -> LABEL(to) -> LABEL(subject) -> VALUE(subject) -> BODY(None)


In [12]:
_LABEL_RE = re.compile(
    r'^(From|Sent|To|Cc|Ce|Subject)\s*:\s*$', re.IGNORECASE
)
_LABEL_VALUE_RE = re.compile(
    r'^(From|Sent|To|Cc|Ce|Subject)\s*:\s+(.+)', re.IGNORECASE
)
_ANY_LABEL_RE = re.compile(
    r'^(From|Sent|To|Cc|Ce|Subject)\s*:', re.IGNORECASE
)


# OCR label normalization: Ce -> Cc, Subject : -> Subject
_LABEL_ALIAS = {"ce": "cc"}


def _find_from_line(lines):
    """Return index of the first 'From:' line (skipping boilerplate)."""
    for i, line in enumerate(lines[:20]):
        if re.match(r'^From[:\s]', line.strip(), re.IGNORECASE):
            return i
    return None


def match_template(template, text):
    """Walk a template structure against text lines.

    Returns a dict of extracted fields, or None if the template doesn't match.
    """
    lines = text.split("\n")
    start = _find_from_line(lines)
    if start is None:
        return None

    extracted = {}
    i = start
    structure = template["structure"]

    for step_idx, (role, field) in enumerate(structure):
        # Skip blank lines
        while i < len(lines) and not lines[i].strip():
            i += 1
        if i >= len(lines):
            return None

        stripped = lines[i].strip()

        if role == Role.LABEL:
            m = _LABEL_RE.match(stripped)
            label = _LABEL_ALIAS.get(m.group(1).lower(), m.group(1).lower()) if m else None
            if label != field:
                return None  # wrong label or has inline value
            i += 1

        elif role == Role.VALUE:
            if _ANY_LABEL_RE.match(stripped):
                return None  # got a label line where we expected a value
            # Collect the primary value line
            parts = [stripped]
            i += 1
            # Only consume continuation lines if the NEXT structural step is
            # another LABEL (e.g. multi-line To/Cc list).  If the next step is
            # BODY we stop here — otherwise we eat body text into the subject
            # field and leave nothing for the BODY step to record.
            next_role = structure[step_idx + 1][0] if step_idx + 1 < len(structure) else None
            if next_role == Role.LABEL:
                while i < len(lines):
                    s = lines[i].strip()
                    if not s:
                        break
                    if _ANY_LABEL_RE.match(s):
                        break
                    parts.append(s)
                    i += 1
            extracted[field] = " ".join(parts)

        elif role == Role.LABEL_VALUE:
            m = _LABEL_VALUE_RE.match(stripped)
            label = _LABEL_ALIAS.get(m.group(1).lower(), m.group(1).lower()) if m else None
            if label != field:
                return None  # wrong label or no inline value
            extracted[field] = m.group(2).strip()
            i += 1

        elif role == Role.BODY:
            # Advance past blanks to find first body line
            while i < len(lines) and not lines[i].strip():
                i += 1
            extracted["_body_start"] = i if i < len(lines) else None
            break

    extracted["_template"] = template["name"]
    return extracted


The matcher walks the template step by step: if the current line matches what the template expects (the right label, a value where a value should be), it advances. If any step fails, it returns `None` immediately — no partial results.

Run the next cell to try both templates on **E25B1E9BF** — the same file where the greedy extractor silently corrupted the To field. The no-Cc template rejects the file (it hits `Ce:` where it expects `Subject:`), and the Cc template matches correctly — extracting To and Cc as separate fields.

In [13]:
# Demo on the Ce: file from section 5 — the one where greedy extraction failed
ocr_text = strip_boilerplate((TXT_DIR / "E25B1E9BF.txt").read_text(encoding="utf-8"))

print("Text fed to templates (first 300 chars):")
print(ocr_text[:300])
print()

# The no-Cc template rejects: Ce: appears where Subject: is expected
result_ncc = match_template(TEMPLATE_ALT_FSTX, ocr_text)
print(f"alt_FStX  (no Cc) -> {result_ncc}")
print("  Rejected — Ce: appears where Subject: is expected.")
print()

# The Cc template succeeds: Ce: is recognized as Cc:
result_cc = match_template(TEMPLATE_ALT_FSTCX, ocr_text)
print(f"alt_FStCX (with Cc) -> matched!")
print(f"  from:    {result_cc['from']!r}")
print(f"  to:      {result_cc['to'][:60]!r}")
print(f"  cc:      {result_cc['cc'][:60]!r}")
print(f"  subject: {result_cc['subject']!r}")

Text fed to templates (first 300 chars):
From: 
Hitchcock, Dorie <dorie.hitchcock@enron.com> 
Sent: 
Fri, 20 Jul 2001 11:13:22 -0700 (PDT) 
To: 
Kitchen, Louise <louise.kitchen@enron.com>, Lavorato, John 
<john.lavorato@enron.com> 
Ce: 
Schoppe, Tammie <tammie.schoppe@enron.com>, Hillis, Kimberly 
<kimberly.hillis@enron.com> 
Subject: 
EA 

alt_FStX  (no Cc) -> None
  Rejected — Ce: appears where Subject: is expected.

alt_FStCX (with Cc) -> matched!
  from:    'Hitchcock, Dorie <dorie.hitchcock@enron.com>'
  to:      'Kitchen, Louise <louise.kitchen@enron.com>, Lavorato, John <'
  cc:      'Schoppe, Tammie <tammie.schoppe@enron.com>, Hillis, Kimberly'
  subject: 'EA EVENT MASTER SCHEDULE'


## 8. Templates as rejection tools

A template doesn't just extract — it **refuses** to extract when the structure doesn't match. Compare this with the greedy extractor from section 4, which silently produced wrong data on E25B1E9BF.

Below, each email is tried against the wrong template first (rejected), then the correct one (matched). A no-Cc email is rejected by the Cc template. The Ce: email is rejected by the no-Cc template but matched by the Cc template — because `Ce:` is recognized as OCR for `Cc:`.

In [14]:
# --- No-Cc email against both templates ---
no_cc_text = strip_boilerplate(txt_files[1].read_text(encoding="utf-8"))

print("Email 1 — no Cc (E0000D1D0):")
print(no_cc_text.split("\n")[0:8])
print()

wrong = match_template(TEMPLATE_ALT_FSTCX, no_cc_text)
print(f"  alt_FStCX (Cc template) -> {wrong}")
print("  Rejected — the email has no Cc: so the Cc template refuses to match.")

right = match_template(TEMPLATE_ALT_FSTX, no_cc_text)
print(f"  alt_FStX  (no-Cc template) -> from={right['from'][:40]!r}, subject={right['subject'][:40]!r}")
print()

# --- Ce: email against both templates ---
ce_text = strip_boilerplate((TXT_DIR / "E25B1E9BF.txt").read_text(encoding="utf-8"))

print(f"Email 2 — with Ce: (E25B1E9BF):")
print(ce_text.split("\n")[0:12])
print()

wrong2 = match_template(TEMPLATE_ALT_FSTX, ce_text)
print(f"  alt_FStX  (no-Cc template) -> {wrong2}")
print("  Rejected — Ce: appears where alt_FStX expects Subject:.")

right2 = match_template(TEMPLATE_ALT_FSTCX, ce_text)
print(f"  alt_FStCX (Cc template)    -> from={right2['from'][:40]!r}, cc={right2['cc'][:40]!r}")

Email 1 — no Cc (E0000D1D0):
['From:', 'EDIS Email Service <edismail@incident.com>', 'Sent:', 'Fri, 5 Apr 2002 00:07:00 -0800 (PST)', 'To:', 'Motley, Matt <matt.motley@enron.com>', 'Subject:', '[EDIS]  EQ 4 2 SAN BERNARDINO COUNTY [News: Statewide]']

  alt_FStCX (Cc template) -> None
  Rejected — the email has no Cc: so the Cc template refuses to match.
  alt_FStX  (no-Cc template) -> from='EDIS Email Service <edismail@incident.co', subject='[EDIS]  EQ 4 2 SAN BERNARDINO COUNTY [Ne'

Email 2 — with Ce: (E25B1E9BF):
['From: ', 'Hitchcock, Dorie <dorie.hitchcock@enron.com> ', 'Sent: ', 'Fri, 20 Jul 2001 11:13:22 -0700 (PDT) ', 'To: ', 'Kitchen, Louise <louise.kitchen@enron.com>, Lavorato, John ', '<john.lavorato@enron.com> ', 'Ce: ', 'Schoppe, Tammie <tammie.schoppe@enron.com>, Hillis, Kimberly ', '<kimberly.hillis@enron.com> ', 'Subject: ', 'EA EVENT MASTER SCHEDULE ']

  alt_FStX  (no-Cc template) -> None
  Rejected — Ce: appears where alt_FStX expects Subject:.
  alt_FStCX (Cc templa

## 9. How the 3 starter templates were built

The three templates in `helpers/enron_templates.py` were built by the same process you just saw:

1. Strip boilerplate from every file
2. Classify each line (label, value, junk)
3. Group files by their role sequence
4. Write one template per unique sequence, ordered by frequency

The top three cover 98.5% of the corpus:

| Template | Pattern | Coverage |
|----------|---------|----------|
| `alt_FStX` | From/Sent/To/Subject | ~72% |
| `alt_FStCX` | From/Sent/To/Cc/Subject | ~25% |
| `alt_Fs_X` | From/Sent/To(empty)/Subject | ~2% |

The remaining ~1.5% (~70 emails) have layouts these three templates reject. You'll explore those failures in the next section, and in lesson 7 you'll build additional templates to cover them.

## 10. Running with 3 templates

`helpers/enron_templates.py` uses the same Role/template/matcher architecture, plus OCR-aware label patterns, junk line skipping, date split handling, and post-match validation.

Run the next cell to import it and apply it to the full corpus.

In [15]:
from helpers.enron_templates import (
    extract_enron_headers,
    strip_enron_boilerplate,
    match_template as enron_match_template,
    _find_from_line as enron_find_from_line,
)
print("Imported template system.")


Imported template system.


In [16]:
import time

parsed = []
failures = []

t0 = time.time()
for f in tqdm(txt_files, desc="extract_enron_headers"):
    text = f.read_text(encoding="utf-8")
    result = extract_enron_headers(text)
    if result is not None:
        parsed.append({
            "doc_id":    f.stem,
            "from":      result.get("from", ""),
            "sent":      result.get("sent", ""),
            "to":        result.get("to", ""),
            "cc":        result.get("cc", ""),
            "subject":   result.get("subject", ""),
            "_template": result["_template"],
        })
    else:
        failures.append(f.stem)

elapsed = time.time() - t0
print(f"\nParsed:  {len(parsed):,} / {len(txt_files):,}")
print(f"Failed:  {len(failures):,} / {len(txt_files):,}")
print(f"Time:    {elapsed:.1f}s ({len(txt_files)/elapsed:,.0f} emails/sec)")


extract_enron_headers:   0%|          | 0/4911 [00:00<?, ?it/s]

extract_enron_headers: 100%|██████████| 4911/4911 [00:00<00:00, 5332.93it/s]


Parsed:  4,841 / 4,911
Failed:  70 / 4,911
Time:    0.9s (5,264 emails/sec)


In [17]:
# Field coverage breakdown
field_counts = {k: 0 for k in ("from", "sent", "to", "cc", "subject")}
for r in parsed:
    for field in field_counts:
        if r.get(field, "").strip():
            field_counts[field] += 1

print("Field coverage:")
for field, count in field_counts.items():
    print(f"  {field:8s}: {count:,} / {len(parsed):,} ({count/len(parsed)*100:.1f}%)")

print()

# Template distribution
template_counts = Counter(r["_template"] for r in parsed)
print("Top templates:")
for t, c in template_counts.most_common(5):
    print(f"  {t:20s} {c:,} ({c/len(parsed)*100:.1f}%)")


Field coverage:
  from    : 4,841 / 4,841 (100.0%)
  sent    : 4,841 / 4,841 (100.0%)
  to      : 4,755 / 4,841 (98.2%)
  cc      : 1,213 / 4,841 (25.1%)
  subject : 4,841 / 4,841 (100.0%)

Top templates:
  alt_FStX             3,542 (73.2%)
  alt_FStCX            1,213 (25.1%)
  alt_Fs_X             86 (1.8%)


In [18]:
# Show 3 sample parsed records
print("Sample parsed records:")
for r in parsed[:3]:
    print(f"\n  doc_id:   {r['doc_id']}")
    print(f"  template: {r['_template']}")
    print(f"  from:     {r['from'][:70]!r}")
    print(f"  sent:     {r['sent'][:70]!r}")
    print(f"  to:       {r['to'][:70]!r}")
    print(f"  subject:  {r['subject'][:70]!r}")


Sample parsed records:

  doc_id:   E0000813E
  template: alt_FStCX
  from:     'Rob Bradley <rob.bradley@enron.com>'
  sent:     'Tue, 21 Nov 2000 08:34:00 -0800 (PST)'
  to:       'Kenneth Lay <kenneth.lay@enron.com>'
  subject:  'Remarks to EES Employees--December 1'

  doc_id:   E0000D1D0
  template: alt_FStX
  from:     'EDIS Email Service <edismail@incident.com>'
  sent:     'Fri, 5 Apr 2002 00:07:00 -0800 (PST)'
  to:       'Motley, Matt <matt.motley@enron.com>'
  subject:  '[EDIS]  EQ 4 2 SAN BERNARDINO COUNTY [News: Statewide]'

  doc_id:   E000B745B
  template: alt_FStCX
  from:     'Susan Scott <susan.scott@enron.com>'
  sent:     'Wed, 31 May 2000 02:24:00 -0700 (PDT)'
  to:       '[REDACTED] B6'
  subject:  'Re: Transwestern CAF Request'


In [19]:
# Save parsed records
out_path = Path("data/parsed_records_enron.json")
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    json.dump(parsed, f, indent=2, ensure_ascii=False)

print(f"Saved {len(parsed):,} records to {out_path}")

# Save failures
fail_path = Path("data/parse_failures_enron.json")
with open(fail_path, "w") as f:
    json.dump(failures, f, indent=2)
print(f"Saved {len(failures):,} failures to {fail_path}")


Saved 4,841 records to data/parsed_records_enron.json
Saved 70 failures to data/parse_failures_enron.json


## 11. Build your own template

The ~70 failures are emails with layouts the three starter templates don't cover -- missing Subject lines, empty Cc fields, unusual field orderings, or OCR-garbled labels that break the matcher.

The cells below load the failure files and let you inspect them. Your task: identify the pattern, write a template definition, and test it. In the next lesson, you'll work with an LLM to speed this up.

> **Note:** The matcher collects multi-line values (like long recipient lists) only when the next step in the template is a LABEL. If you create a template where VALUE follows VALUE, only the first line of each value is captured. Ensure every VALUE step for a multi-line field (To, Cc) is followed by a LABEL step.

In [ ]:
# ---------------------------------------------------------------
# Step 1: Pick a failure and read its stripped text
# ---------------------------------------------------------------
# (run section 10 first so `failures` is defined)

if failures:
    target_id = failures[0]  # change this index to inspect a different failure
    target_path = TXT_DIR / f"{target_id}.txt"
    target_raw = target_path.read_text(encoding="utf-8")
    target_stripped = strip_enron_boilerplate(target_raw)

    print(f"Doc ID: {target_id}")
    print()
    print("Stripped text (paste this into an LLM chat along with the")
    print("template examples above, and ask it to generate a template):")
    print("-" * 50)
    print(target_stripped[:800])
    print("-" * 50)

In [21]:
# ---------------------------------------------------------------
# Step 2: Paste the template dict from the conversation lesson and test it
# ---------------------------------------------------------------

# Replace this with the template the LLM returns
MY_TEMPLATE = {
    "name": "my_template",
    'structure': [
        (LV, 'from'), (LV, 'sent'), (LV, 'to'),
        (LV, 'cc'), (LV, 'subject'), (B, None),
    ],
}

# Test it on the target file
if failures and MY_TEMPLATE["structure"]:
    result = enron_match_template(MY_TEMPLATE, target_stripped.split('\n'),
                                  enron_find_from_line(target_stripped.split('\n')) or 0)
    print(f"Result on {target_id}:")
    print(result)
    print()

    # Check how many other failures it also resolves
    also_resolved = []
    for doc_id in failures[:100]:  # check first 100 failures
        path = TXT_DIR / f"{doc_id}.txt"
        if path.exists():
            raw = path.read_text(encoding="utf-8")
            stripped = strip_enron_boilerplate(raw)
            lines = stripped.split('\n')
            start = enron_find_from_line(lines)
            if start is not None:
                r = enron_match_template(MY_TEMPLATE, lines, start)
                if r is not None:
                    also_resolved.append(doc_id)

    print(f"Also resolves {len(also_resolved)} of the first {min(100, len(failures))} failures")
else:
    print("Run section 10 first, or add a structure to MY_TEMPLATE")


Result on E076559BC:
None

Also resolves 0 of the first 70 failures


## Summary

- **Boilerplate stripping** clears the document header so field extraction starts at a predictable position
- **The alternating pattern** (label on one line, value on the next) is consistent across the corpus — the variation is in which fields appear
- A **greedy extractor** works on clean files but silently produces wrong data when OCR corrupts labels — `Ce:` absorbed into the `To` field with no indication of error
- **Templates** encode exact role sequences and refuse to match anything that deviates — clean rejection instead of silent corruption
- **OCR label aliases** (`Ce:` → `Cc:`, `Subject :` with extra space) let templates handle predictable OCR variants
- Three starter templates cover **98.5%** of the corpus at ~2,500 emails/sec
- The **~70 failures** are edge cases with unusual layouts — targets for new templates or downstream tools